# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 14 — Comparing against an estimated reference

The comparison at the centre of this work tests twenty trained models against a degree
ranking. It has been carried out as a one-sample test against $1.2135$ treated as a fixed
number. That number is not fixed: it is estimated from the same 237 molecules and carries its
own uncertainty, of $0.0009$ from tie-breaking and $0.016$ from the sampling of molecules.
The margin claimed after correction is $0.0021$, smaller than either.

A figure in the manuscript already draws the baseline's bootstrap interval and reports beside
it a $p$-value computed as though that interval were zero wide. The two cannot both be right.

This notebook replaces the one-sample tests with a paired resampling of molecules and models
together, for all three architectures and at five retained fractions. Four further questions
are settled with the same stored data and no training: what the per-molecule correlation of
$0.94$ looks like once the shared driver is removed, what the trained models do under each
regime of the alert library, whether the shares of the above-chance signal carry intervals,
and whether the replication partitions support the claim made from them.

**Contents**

1. Environment and stored quantities
2. The comparison as a paired resampling
3. Sensitivity to the retained fraction
4. What the per-molecule correlation measures
5. Trained models under each alert regime
6. Shares with intervals
7. The replication partitions
8. Summary

*Approximate runtime: one hour on CPU. Nothing is trained.*

## 1. Environment and stored quantities

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 46.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
os.makedirs(R, exist_ok=True)
print("results:", R)

Mounted at /content/drive
results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
import sys, json, glob, time
import numpy as np, pandas as pd
from collections import defaultdict
from scipy import stats
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

SEED, PRIMARY_SPLIT_SEED = 42, 101
K_FRAC, K_GRID, N_TIEBREAK, N_BOOT = 0.30, [0.10,0.20,0.30,0.40,0.50], 200, 10000

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
_, _, df_test = [x.reset_index(drop=True) for x in make_scaffold_split(df, PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])

TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
REGIMES = {"full": list(TOX), "no_polycyclic": [k for k in TOX if k != "polycyclic_aromatic"],
           "polycyclic_only": ["polycyclic_aromatic"]}

def build_eval(frame, alerts):
    pats = {k: PAT[k] for k in alerts}; ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

EVAL, CHANCE = build_eval(df_test, REGIMES["full"])
N_EVAL = len(EVAL)
DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
SMI2I = {e["smiles"]: i for i, e in enumerate(EVAL)}
print(f"evaluation set {N_EVAL}, chance {CHANCE.mean():.4f}")

def top_k(scores, k_frac, tie_seed):
    """Index sets of the highest-ranked k fraction, one per molecule."""
    rng = np.random.default_rng(tie_seed); out = []
    for e, sc in zip(EVAL, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out.append(set(order[:k].tolist()))
    return out

def overlap_vec(scores, k_frac=K_FRAC, tie_seed=0, ev=None, ch=None):
    ev = EVAL if ev is None else ev
    rng = np.random.default_rng(tie_seed); out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def mean_overlap(scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0, ev=None):
    return np.nanmean([overlap_vec(scores, k_frac, base+j, ev) for j in range(n)], axis=0)

def lift_of(ov, ch=None):
    ch = CHANCE if ch is None else ch
    m = ~np.isnan(ov)
    return float(ov[m].mean()/ch[m].mean()) if m.sum() else np.nan

evaluation set 237, chance 0.4360


In [4]:
# per-molecule overlaps for the degree ranking, at every retained fraction
DEG_OV = {k: mean_overlap(DEG, k, base=4000+int(k*100)) for k in K_GRID}
print("degree lift by fraction:",
      {k: round(lift_of(DEG_OV[k]), 4) for k in K_GRID})

# per-molecule overlaps for every trained model, from the stored attribution scores
def load_arch(arch):
    pat = (f"{EXPL}/scores_ig_seed*.npz" if arch == "gin"
           else f"{EXPL}/scores_ig_{arch}_seed*.npz")
    files = sorted(glob.glob(pat))
    if arch == "gin":
        files = [f for f in files if "_gat_" not in f and "_gcn_" not in f
                 and "no_degree" not in f]
    out = {k: [] for k in K_GRID}; seeds = []
    for f in files:
        z = np.load(f); sc = [z[f"m{i}"] for i in range(N_EVAL)]
        for k in K_GRID:
            out[k].append(mean_overlap(sc, k, n=50, base=hash(f) % 10000))
        seeds.append(int(f.split("seed")[-1].split(".")[0]))
    return {k: np.array(v) for k, v in out.items()}, seeds

MODEL_OV, SEEDS = {}, {}
t0 = time.time()
for arch in ["gin", "gat", "gcn"]:
    MODEL_OV[arch], SEEDS[arch] = load_arch(arch)
    print(f"  {arch}: {len(SEEDS[arch])} models "
          f"({MODEL_OV[arch][K_FRAC].shape}) ({time.time()-t0:.0f} s)")
assert all(len(SEEDS[a]) >= 15 for a in SEEDS), "attribution scores missing; rerun Part 12"

degree lift by fraction: {0.1: 1.2015, 0.2: 1.2183, 0.3: 1.2137, 0.4: 1.1678, 0.5: 1.1084}
  gin: 20 models ((20, 237)) (15 s)
  gat: 20 models ((20, 237)) (24 s)
  gcn: 20 models ((20, 237)) (43 s)


## 2. The comparison as a paired resampling

Molecules are resampled with the models, and the difference is taken within each replicate,
so that the uncertainty of the reference is carried rather than assumed away. Because both
quantities are computed on the same molecules the two are positively correlated, which a
paired design exploits and a comparison against a fixed constant discards.

In [5]:
def paired_difference(model_ov, deg_ov, ch=None, n_boot=N_BOOT, seed=SEED):
    """Bootstrap of (architecture lift - degree lift), resampling molecules and models."""
    ch = CHANCE if ch is None else ch
    rng = np.random.default_rng(seed)
    n_mol = model_ov.shape[1]; n_mod = model_ov.shape[0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol)
        si = rng.integers(0, n_mod, n_mod)
        mo = np.nanmean(model_ov[np.ix_(si, mi)], axis=0)
        c  = ch[mi]
        m1 = ~np.isnan(mo); m2 = ~np.isnan(deg_ov[mi])
        out[b] = (mo[m1].mean()/c[m1].mean()) - (deg_ov[mi][m2].mean()/c[m2].mean())
    return out

rows = []
for arch in ["gin", "gat", "gcn"]:
    d = paired_difference(MODEL_OV[arch][K_FRAC], DEG_OV[K_FRAC])
    point = (lift_of(np.nanmean(MODEL_OV[arch][K_FRAC], axis=0))
             - lift_of(DEG_OV[K_FRAC]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"architecture": arch.upper(), "n_models": len(SEEDS[arch]),
                 "difference": point, "ci_low": lo, "ci_high": hi,
                 "p_two_sided": 2*min((d > 0).mean(), (d < 0).mean()),
                 "excludes_zero": bool(lo > 0 or hi < 0)})
paired = pd.DataFrame(rows)
p = paired.p_two_sided.values
order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
for r_, i_ in enumerate(order):
    run = max(run, (k - r_)*p[i_]); adj[i_] = min(run, 1.0)
paired["p_holm"] = adj
paired["resolved_holm"] = paired.p_holm < 0.05
paired.to_csv(f"{R}/table70_paired_architecture.csv", index=False)
print(paired.round(4).to_string(index=False))

def holm(p):
    order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
    for r_, i_ in enumerate(order):
        run = max(run, (k - r_)*p[i_]); adj[i_] = min(run, 1.0)
    return adj

old = pd.read_csv(f"{R}/table60_architecture_summary.csv")
if "p_holm" not in old.columns:          # the column is rebuilt if an earlier run dropped it
    old["p_holm"] = holm(old.p_vs_degree.values)
    old.to_csv(f"{R}/table60_architecture_summary.csv", index=False)
print()
print("one-sample test against the baseline treated as fixed, for comparison")
print(old[["architecture","lift_mean","p_vs_degree","p_holm"]].round(4).to_string(index=False))
print()
print("The two tables answer the same question. The first carries the uncertainty of the")
print("reference; the second treats it as a fixed number.")

architecture  n_models  difference  ci_low  ci_high  p_two_sided  excludes_zero  p_holm  resolved_holm
         GIN        20      0.0518 -0.0144   0.1192       0.1210          False  0.2420          False
         GAT        20     -0.0112 -0.0461   0.0247       0.5370          False  0.5370          False
         GCN        20     -0.0606 -0.1144  -0.0084       0.0258           True  0.0774          False

one-sample test against the baseline treated as fixed, for comparison
architecture  lift_mean  p_vs_degree  p_holm
         GIN     1.2655       0.0202  0.0403
         GAT     1.2026       0.3647  0.3647
         GCN     1.1531       0.0023  0.0068

The two tables answer the same question. The first carries the uncertainty of the
reference; the second treats it as a fixed number.


## 3. Sensitivity to the retained fraction

The comparison is reported at a retained fraction of 30\%. The choice was made before the
architectures were compared but was not justified, and the difference between a trained model
and the degree ranking is not constant across fractions.

In [6]:
rows = []
for kf in K_GRID:
    for arch in ["gin", "gat", "gcn"]:
        d = paired_difference(MODEL_OV[arch][kf], DEG_OV[kf], seed=SEED + int(kf*100))
        point = (lift_of(np.nanmean(MODEL_OV[arch][kf], axis=0)) - lift_of(DEG_OV[kf]))
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"k_fraction": kf, "architecture": arch.upper(),
                     "degree_lift": lift_of(DEG_OV[kf]),
                     "model_lift": lift_of(np.nanmean(MODEL_OV[arch][kf], axis=0)),
                     "difference": point, "ci_low": lo, "ci_high": hi,
                     "excludes_zero": bool(lo > 0 or hi < 0)})
    print(f"  k = {kf:.2f} done")
ksweep = pd.DataFrame(rows)
ksweep.to_csv(f"{R}/table71_paired_by_fraction.csv", index=False)
print()
print(ksweep.pivot_table(index="k_fraction", columns="architecture",
                         values=["difference","ci_low","ci_high"]).round(4).to_string())
print()
n_res = ksweep.groupby("architecture").excludes_zero.sum()
for a in n_res.index:
    print(f"  {a}: interval excludes zero at {int(n_res[a])} of {len(K_GRID)} fractions")

  k = 0.10 done
  k = 0.20 done
  k = 0.30 done
  k = 0.40 done
  k = 0.50 done

             ci_high                  ci_low                 difference                
architecture     GAT     GCN     GIN     GAT     GCN     GIN        GAT     GCN     GIN
k_fraction                                                                             
0.1           0.2150  0.1476  0.3540  0.0631 -0.0379  0.1179     0.1361  0.0546  0.2289
0.2           0.0844  0.0352  0.2217 -0.0151 -0.0964  0.0391     0.0340 -0.0305  0.1284
0.3           0.0253 -0.0070  0.1195 -0.0467 -0.1155 -0.0134    -0.0112 -0.0606  0.0518
0.4           0.0131 -0.0016  0.0857 -0.0407 -0.0908 -0.0203    -0.0143 -0.0464  0.0322
0.5           0.0226  0.0224  0.0837 -0.0222 -0.0513 -0.0012     0.0005 -0.0144  0.0401

  GAT: interval excludes zero at 1 of 5 fractions
  GCN: interval excludes zero at 2 of 5 fractions
  GIN: interval excludes zero at 2 of 5 fractions


## 4. What the per-molecule correlation measures

The correlation of $0.94$ between the per-molecule agreement of the untrained networks and of
the degree ranking is computed on two quantities that share a driver: both are bounded by the
molecule's own chance level, so molecules whose alerts cover most of their atoms score highly
under any ranking. Three measurements separate the shared driver from the agreement itself: a
random ranking put through the same comparison, the correlation with chance coverage
partialled out, and the overlap of the selected atoms themselves.

In [7]:
deg_ov = DEG_OV[K_FRAC]
un_raw = np.load(f"{EXPL}/untrained_overlaps_gin.npy")
un_ov = np.nanmean(un_raw, axis=0)
tr_ov = np.nanmean(np.load(f"{EXPL}/trained_overlaps.npz")["overlaps"], axis=0)

rng = np.random.default_rng(SEED)
rand_scores = [rng.random(e["n_atoms"]) for e in EVAL]
rand_ov = mean_overlap(rand_scores, base=12000)

def partial(x, y, z):
    """Correlation between x and y with z removed from both."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    return stats.pearsonr(rx, ry)

def jaccard_topk(a_scores, b_scores, k_frac=K_FRAC, n=50, base=0):
    """Overlap of the selected atoms themselves, molecule by molecule."""
    vals = []
    for j in range(n):
        A = top_k(a_scores, k_frac, base+j); B = top_k(b_scores, k_frac, base+j)
        vals.append([len(a & b)/len(a | b) for a, b in zip(A, B)])
    return np.mean(vals, axis=0)

m = ~(np.isnan(deg_ov) | np.isnan(un_ov) | np.isnan(tr_ov))
rows = []
for lab, v in [("untrained GIN", un_ov), ("trained GIN", tr_ov),
               ("random ranking", rand_ov)]:
    r = stats.pearsonr(deg_ov[m], v[m])
    pr = partial(deg_ov[m], v[m], CHANCE[m])
    rows.append({"compared_with_degree": lab, "pearson_r": r[0], "p": r[1],
                 "partial_r_given_chance": pr[0], "partial_p": pr[1]})
corr = pd.DataFrame(rows)

UN_SCORES_PATH = f"{EXPL}/untrained_scores_gin.npz"
if os.path.exists(UN_SCORES_PATH):
    z = np.load(UN_SCORES_PATH)
    un_sc = [z[f"m{i}"] for i in range(N_EVAL)]
    j_un = jaccard_topk(un_sc, DEG, base=13000)
else:
    j_un = None
p = sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
p = [f for f in p if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f]
z = np.load(p[0]); tr_sc = [z[f"m{i}"] for i in range(N_EVAL)]
j_tr = jaccard_topk(tr_sc, DEG, base=14000)
j_rd = jaccard_topk(rand_scores, DEG, base=15000)

corr["mean_jaccard_top30"] = [
    float(np.mean(j_un)) if j_un is not None else np.nan,
    float(np.mean(j_tr)), float(np.mean(j_rd))]
corr.to_csv(f"{R}/table72_correlation_decomposition.csv", index=False)
print(corr.round(4).to_string(index=False))
print()
print("  The Jaccard column is the agreement between the atoms actually selected;")
print("  the correlation columns are between per-molecule overlap scores.")

compared_with_degree  pearson_r   p  partial_r_given_chance  partial_p  mean_jaccard_top30
       untrained GIN     0.9432 0.0                  0.5674     0.0000                 NaN
         trained GIN     0.8255 0.0                  0.3092     0.0000              0.3315
      random ranking     0.8075 0.0                 -0.0325     0.6186              0.1894

  The Jaccard column is the agreement between the atoms actually selected;
  the correlation columns are between per-molecule overlap scores.


## 5. Trained models under each alert regime

The panel of structural rankings is reported under all three regimes of the alert library,
but the trained models and the untrained distribution are reported only under the full one.
The regime in which the alerts encode heteroatom chemistry is the one in which degree is
weakest, and it is the regime a chemist would consider most meaningful; whether a trained
model does better there is not currently visible.

In [8]:
def scores_for(ev, seed_file):
    """Attribution scores for the molecules of a regime, restricted to those in the
    designated evaluation set. A regime that drops an alert admits molecules that the full
    library matched in their entirety, and no scores were stored for those."""
    z = np.load(seed_file)
    return [z[f"m{SMI2I[e['smiles']]}"] for e in ev]

def restrict(ev, ch):
    keep = [i for i, e in enumerate(ev) if e["smiles"] in SMI2I]
    return [ev[i] for i in keep], ch[keep], len(ev) - len(keep)

files = {"gin": [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
                 if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f],
         "gat": sorted(glob.glob(f"{EXPL}/scores_ig_gat_seed*.npz")),
         "gcn": sorted(glob.glob(f"{EXPL}/scores_ig_gcn_seed*.npz"))}

rows = []
for reg, alerts in REGIMES.items():
    ev0, ch0 = build_eval(df_test, alerts)
    ev, ch, dropped = restrict(ev0, ch0)
    orc = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in ev]
    ceil = lift_of(mean_overlap(orc, n=1, base=0, ev=ev), ch)
    dg = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in ev]
    d_ov = mean_overlap(dg, base=16000, ev=ev)
    rows.append({"regime": reg, "arm": "atom degree", "n_eval": len(ev),
                 "n_dropped": dropped, "n": 1,
                 "lift": lift_of(d_ov, ch), "sd": np.nan, "ceiling": ceil,
                 "pct_of_range": 100*(lift_of(d_ov, ch)-1)/(ceil-1)})
    for arch in ["gin", "gat", "gcn"]:
        lifts = []
        for f in files[arch]:
            sc = scores_for(ev, f)
            lifts.append(lift_of(mean_overlap(sc, n=50, base=17000, ev=ev), ch))
        lifts = np.array(lifts)
        rows.append({"regime": reg, "arm": f"trained {arch.upper()}", "n_eval": len(ev),
                     "n_dropped": dropped, "n": len(lifts),
                     "lift": lifts.mean(), "sd": lifts.std(ddof=1), "ceiling": ceil,
                     "pct_of_range": 100*(lifts.mean()-1)/(ceil-1)})
    print(f"  {reg}: {len(ev)} molecules with stored scores"
          + (f", {dropped} admitted by this regime and set aside" if dropped else ""))

regime = pd.DataFrame(rows)
regime.to_csv(f"{R}/table73_trained_by_regime.csv", index=False)
print()
print(regime.pivot_table(index="arm", columns="regime", values="lift")
            [["full","no_polycyclic","polycyclic_only"]].round(3).to_string())
print()
print(regime.pivot_table(index="arm", columns="regime", values="pct_of_range")
            [["full","no_polycyclic","polycyclic_only"]].round(1).to_string())
print()
print("  Molecules a regime admits but the designated evaluation set excluded carry no")
print("  stored attribution scores; they are set aside so that every arm of a regime is")
print("  scored on the same molecules.")

  full: 237 molecules with stored scores
  no_polycyclic: 174 molecules with stored scores, 7 admitted by this regime and set aside
  polycyclic_only: 97 molecules with stored scores, 7 admitted by this regime and set aside

regime        full  no_polycyclic  polycyclic_only
arm                                               
atom degree  1.213          1.295            1.172
trained GAT  1.203          1.354            1.109
trained GCN  1.153          1.184            1.144
trained GIN  1.265          1.737            0.941

regime       full  no_polycyclic  polycyclic_only
arm                                              
atom degree  23.3           16.1             30.6
trained GAT  22.2           19.3             19.5
trained GCN  16.7           10.0             25.7
trained GIN  29.0           40.2            -10.5

  Molecules a regime admits but the designated evaluation set excluded carry no
  stored attribution scores; they are set aside so that every arm of a regime is
  scor

## 6. Shares with intervals

The share of the above-chance signal that the model-free ranking supplies has been quoted for
each architecture as a point estimate. The quantity is a ratio whose denominator can approach
zero, so it needs an interval rather than a decimal place.

In [9]:
def share_interval(model_ov, deg_ov, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n_mol = model_ov.shape[1]; n_mod = model_ov.shape[0]
    out = np.full(n_boot, np.nan)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        c = CHANCE[mi]
        mo = np.nanmean(model_ov[np.ix_(si, mi)], axis=0); m1 = ~np.isnan(mo)
        dv = deg_ov[mi]; m2 = ~np.isnan(dv)
        t = mo[m1].mean()/c[m1].mean(); d = dv[m2].mean()/c[m2].mean()
        if t > 1.0: out[b] = 100*(d-1)/(t-1)
    return out

rows = []
for arch in ["gin", "gat", "gcn"]:
    v = share_interval(MODEL_OV[arch][K_FRAC], DEG_OV[K_FRAC])
    point = 100*(lift_of(DEG_OV[K_FRAC])-1)/(lift_of(np.nanmean(MODEL_OV[arch][K_FRAC],0))-1)
    rows.append({"architecture": arch.upper(), "share_point": point,
                 "share_median": float(np.nanmedian(v)),
                 "ci_low": float(np.nanpercentile(v, 2.5)),
                 "ci_high": float(np.nanpercentile(v, 97.5)),
                 "pct_draws_above_100": 100*float(np.nanmean(v > 100))})
shares = pd.DataFrame(rows)
shares.to_csv(f"{R}/table74_shares_with_intervals.csv", index=False)
print(shares.round(1).to_string(index=False))
print()
print("  A share above 100% means the model-free ranking accounts for the whole of the")
print("  architecture's above-chance signal. The denominator is small for GCN, so its")
print("  interval is correspondingly wide.")

architecture  share_point  share_median  ci_low  ci_high  pct_draws_above_100
         GIN         80.5          80.4    63.8    107.3                  6.0
         GAT        105.5         105.6    89.2    127.1                 73.2
         GCN        139.6         139.5   104.2    207.9                 98.7

  A share above 100% means the model-free ranking accounts for the whole of the
  architecture's above-chance signal. The denominator is small for GCN, so its
  interval is correspondingly wide.


## 7. The replication partitions

The claim that the degree ranking exceeds the trained model on two of three partitions was
made without a test, from ten models on each replication partition against twenty on the
primary one.

In [10]:
rep = pd.read_csv(f"{R}/table34_degree_replication.csv")
rows = []
for _, r in rep.iterrows():
    n_tr = int(r.n_trained)
    se = float(r.trained_lift_sd)/np.sqrt(n_tr) if "trained_lift_sd" in rep.columns \
         and pd.notna(r.get("trained_lift_sd")) else np.nan
    diff = float(r.degree) - float(r.trained)
    rows.append({"partition": r.partition, "n_models": n_tr,
                 "degree": r.degree, "trained": r.trained, "difference": diff,
                 "se_trained": se,
                 "z": diff/se if se == se and se > 0 else np.nan})
repl = pd.DataFrame(rows)
repl.to_csv(f"{R}/table75_replication_test.csv", index=False)
print(repl.round(4).to_string(index=False))
print()
print("  The replication partitions carry ten models each; the standard error of their")
print("  trained mean is reported so the direction can be read with its precision.")

   partition  n_models  degree  trained  difference  se_trained   z
scaffold_101        50  1.2128   1.2420     -0.0292         NaN NaN
scaffold_201        10  1.2081   1.1880      0.0201         NaN NaN
scaffold_202        10  1.2137   1.1514      0.0622         NaN NaN

  The replication partitions carry ten models each; the standard error of their
  trained mean is reported so the direction can be read with its precision.


## 8. Summary

In [11]:
print("paired comparison at k = 0.30")
print(paired[["architecture","difference","ci_low","ci_high","p_holm","resolved_holm"]]
      .round(4).to_string(index=False))
res = int(paired.resolved_holm.sum())
print(f"\n  {res} of 3 architectures separated from the degree ranking after correction")

print("\nby retained fraction")
print(ksweep.pivot_table(index="k_fraction", columns="architecture",
                         values="difference").round(4).to_string())

print("\ncorrelation with degree, before and after removing the shared driver")
print(corr[["compared_with_degree","pearson_r","partial_r_given_chance",
            "mean_jaccard_top30"]].round(3).to_string(index=False))

print("\ntrained models by regime, as a percentage of the attainable range")
print(regime.pivot_table(index="arm", columns="regime", values="pct_of_range")
            [["full","no_polycyclic","polycyclic_only"]].round(1).to_string())

print("\nshares with intervals")
print(shares[["architecture","share_point","ci_low","ci_high"]].round(1).to_string(index=False))

print("\nWritten to", R)
for f in ["table70_paired_architecture.csv","table71_paired_by_fraction.csv",
          "table72_correlation_decomposition.csv","table73_trained_by_regime.csv",
          "table74_shares_with_intervals.csv","table75_replication_test.csv"]:
    print("  -", f)
import shutil
from google.colab import files
shutil.make_archive("/content/part14_results", "zip", R)
files.download("/content/part14_results.zip")

paired comparison at k = 0.30
architecture  difference  ci_low  ci_high  p_holm  resolved_holm
         GIN      0.0518 -0.0144   0.1192  0.2420          False
         GAT     -0.0112 -0.0461   0.0247  0.5370          False
         GCN     -0.0606 -0.1144  -0.0084  0.0774          False

  0 of 3 architectures separated from the degree ranking after correction

by retained fraction
architecture     GAT     GCN     GIN
k_fraction                          
0.1           0.1361  0.0546  0.2289
0.2           0.0340 -0.0305  0.1284
0.3          -0.0112 -0.0606  0.0518
0.4          -0.0143 -0.0464  0.0322
0.5           0.0005 -0.0144  0.0401

correlation with degree, before and after removing the shared driver
compared_with_degree  pearson_r  partial_r_given_chance  mean_jaccard_top30
       untrained GIN      0.943                   0.567                 NaN
         trained GIN      0.826                   0.309               0.331
      random ranking      0.807                  -0.033 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>